In [19]:
import pandas as pd
import os

BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
# Using your main 2k file as the source
ORIGINAL_CSV = os.path.join(BASE_PATH, "msrvtt_train_2k_fullcaptions.csv") 
POWER_CSV = os.path.join(BASE_PATH, "task3_power_1500.csv")

try:
    df = pd.read_csv(ORIGINAL_CSV)
    df_power = df.head(1500) 
    df_power.to_csv(POWER_CSV, index=False)
    print(f"✅ Power Dataset created with {len(df_power)} videos.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Power Dataset created with 1500 videos.


In [23]:
import torch
import cv2
import pandas as pd
import time
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BlipForConditionalGeneration, AutoProcessor
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "./blip_video_model_2")
DATA_CSV = os.path.join(BASE_PATH, "task3_power_1500.csv")

# Set to match your existing file so we can RESUME!
RESULTS_CSV = os.path.join(BASE_PATH, "task3_gold_results.csv") 
CHECKPOINT_DIR = os.path.join(BASE_PATH, "checkpoints_task3")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

ENCODER_BLOCKS = [12, 10, 8, 6]   
FRAME_COUNTS = [16, 12, 8]     
EPOCHS = 2                 

class MSRVTTDataset(Dataset):
    def __init__(self, csv_file, processor, num_frames):
        self.data = pd.read_csv(csv_file)
        self.processor = processor
        self.num_frames = num_frames

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        vid_path = self.data.iloc[idx]['video_path'].replace("project 2", "video_captioning")
        caption = str(self.data.iloc[idx]['caption'])
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, max(0, total_frames - 1), self.num_frames, dtype=int)
        frames = []
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret: frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)))
        cap.release()
        while len(frames) < self.num_frames: 
            frames.append(frames[-1] if frames else np.zeros((224,224,3), dtype=np.uint8))
        avg_frame = np.mean(frames, axis=0).astype(np.uint8)
        
        # --- THE FIX: Added truncation=True ---
        inputs = self.processor(
            images=avg_frame, 
            text=caption, 
            return_tensors="pt", 
            padding="max_length", 
            truncation=True, # This forces long sentences to clip at 30!
            max_length=30
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}

def fine_tune_decoder(model, dataloader, blocks_count):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"tuned_model_{blocks_count}blocks.pt")
    
    if os.path.exists(checkpoint_path):
        print(f"      📥 Loading existing checkpoint for {blocks_count} blocks...")
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
        return model

    print(f"      ⚙️ Training {blocks_count}-block model for 2 epochs...")
    for param in model.vision_model.parameters(): param.requires_grad = False
    optimizer = torch.optim.AdamW(model.text_decoder.parameters(), lr=5e-5)
    model.train()
    
    for epoch in range(EPOCHS):
        pbar = tqdm(dataloader, desc=f"      Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            pixel_values = batch["pixel_values"].to(DEVICE)
            
            outputs = model(pixel_values=pixel_values, input_ids=input_ids, labels=input_ids)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            
    torch.save(model.state_dict(), checkpoint_path)
    return model

def evaluate(model, dataloader, processor):
    model.eval()
    bleu_scores = []
    start_time = time.time()
    smoothie = SmoothingFunction().method4
    pbar = tqdm(dataloader, desc="      Evaluating", leave=False)
    with torch.no_grad():
        for batch in pbar:
            out_ids = model.generate(pixel_values=batch["pixel_values"].to(DEVICE), max_length=20)
            pred = processor.decode(out_ids[0], skip_special_tokens=True).split()
            ref = [processor.decode(batch["input_ids"][0], skip_special_tokens=True).split()]
            bleu_scores.append(sentence_bleu(ref, pred, smoothing_function=smoothie))
    latency = (time.time() - start_time) / len(dataloader.dataset)
    return np.mean(bleu_scores), latency

def run_task3_power():
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    
    # Check what we already finished!
    if os.path.exists(RESULTS_CSV):
        results_df = pd.read_csv(RESULTS_CSV)
        results = results_df.to_dict('records')
        completed = set((row['Blocks'], row['Frames']) for row in results)
        print(f"🔄 Resuming from previous run. Found {len(completed)} completed configs.")
    else:
        results = []
        completed = set()

    for blocks in ENCODER_BLOCKS:
        print(f"\n🧪 DEPTH: {blocks} BLOCKS")
        model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
        model.vision_model.encoder.layers = model.vision_model.encoder.layers[:blocks]
        model.to(DEVICE)
        
        if blocks < 12:
            train_loader = DataLoader(MSRVTTDataset(DATA_CSV, processor, num_frames=8), batch_size=4, shuffle=True)
            model = fine_tune_decoder(model, train_loader, blocks)

        for frames in FRAME_COUNTS:
            if (blocks, frames) in completed:
                print(f"   ⏩ Skipping {frames} frames (Already Done)")
                continue
                
            print(f"   ▶ {frames} Frames Configuration")
            eval_loader = DataLoader(MSRVTTDataset(DATA_CSV, processor, num_frames=frames), batch_size=1) 
            bleu, latency = evaluate(model, eval_loader, processor)
            
            print(f"      ↳ BLEU: {bleu:.4f} | {latency:.2f}s")
            results.append({"Blocks": blocks, "Frames": frames, "BLEU4": bleu, "Latency": latency})
            pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

    print(f"\n✅ All Finished! Final Data: {RESULTS_CSV}")

if __name__ == "__main__":
    run_task3_power()


🧪 DEPTH: 12 BLOCKS


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 18936.72it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   ▶ 16 Frames Configuration


      ↳ BLEU: 0.0502 | 0.23s
   ▶ 12 Frames Configuration


      ↳ BLEU: 0.0484 | 0.21s
   ▶ 8 Frames Configuration


...

      ⚙️ Training 6-block model for 2 epochs...


   ▶ 16 Frames Configuration


      ↳ BLEU: 0.0912 | 0.23s
   ▶ 12 Frames Configuration


      ↳ BLEU: 0.0931 | 0.20s
   ▶ 8 Frames Configuration


      ↳ BLEU: 0.0935 | 0.18s

✅ All Finished! Final Data: /Users/ayraj/Desktop/video_captioning/task3_gold_results.csv


In [25]:
import torch
import cv2
import pandas as pd
import os
import time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import BlipForConditionalGeneration, AutoProcessor
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "blip_video_model_2")
DATA_CSV = os.path.join(BASE_PATH, "task3_power_1500.csv") 
RESULTS_CSV = os.path.join(BASE_PATH, "task3_gold_results.csv") # Your existing file!
CHECKPOINT_DIR = os.path.join(BASE_PATH, "checkpoints_task3")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
EPOCHS = 2
FRAME_COUNTS = [16, 12, 8] # The configs we need to evaluate

class MSRVTTDataset(Dataset):
    def __init__(self, csv_file, processor, num_frames):
        self.data = pd.read_csv(csv_file)
        self.processor = processor
        self.num_frames = num_frames

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        vid_path = self.data.iloc[idx]['video_path'].replace("project 2", "video_captioning")
        caption = str(self.data.iloc[idx]['caption'])
        
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, max(0, total_frames - 1), self.num_frames, dtype=int)
        frames = []
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret: frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224)))
        cap.release()
        
        while len(frames) < self.num_frames: 
            frames.append(frames[-1] if frames else np.zeros((224,224,3), dtype=np.uint8))
            
        avg_frame = np.mean(frames, axis=0).astype(np.uint8)
        
        inputs = self.processor(
            images=avg_frame, text=caption, return_tensors="pt", 
            padding="max_length", truncation=True, max_length=30
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}

def evaluate(model, dataloader, processor):
    model.eval()
    bleu_scores = []
    start_time = time.time()
    smoothie = SmoothingFunction().method4
    pbar = tqdm(dataloader, desc="      Evaluating", leave=False)
    with torch.no_grad():
        for batch in pbar:
            out_ids = model.generate(pixel_values=batch["pixel_values"].to(DEVICE), max_length=20)
            pred = processor.decode(out_ids[0], skip_special_tokens=True).split()
            ref = [processor.decode(batch["input_ids"][0], skip_special_tokens=True).split()]
            bleu_scores.append(sentence_bleu(ref, pred, smoothing_function=smoothie))
    latency = (time.time() - start_time) / len(dataloader.dataset)
    return np.mean(bleu_scores), latency

def run_12_block_full():
    print(f"🚀 Launching 12-Block Train & Evaluate on: {DEVICE}")
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, "tuned_model_12blocks.pt")
    
    # --- 1. TRAIN THE MODEL ---
    for param in model.vision_model.parameters(): param.requires_grad = False
    optimizer = torch.optim.AdamW(model.text_decoder.parameters(), lr=5e-5)
    
    train_loader = DataLoader(MSRVTTDataset(DATA_CSV, processor, num_frames=8), batch_size=4, shuffle=True)
    model.train()
    print("   ⚙️ Training 12-block model for 2 epochs...")
    
    for epoch in range(EPOCHS):
        pbar = tqdm(train_loader, desc=f"      Epoch {epoch+1}/{EPOCHS}")
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            pixel_values = batch["pixel_values"].to(DEVICE)
            
            outputs = model(pixel_values=pixel_values, input_ids=input_ids, labels=input_ids)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            
    torch.save(model.state_dict(), checkpoint_path)
    print(f"   ✅ Checkpoint saved to: {checkpoint_path}")

    # --- 2. EVALUATE ON THE 3 FRAME CONFIGS ---
    new_results = []
    for frames in FRAME_COUNTS:
        print(f"\n   ▶ Evaluating {frames} Frames Configuration")
        eval_loader = DataLoader(MSRVTTDataset(DATA_CSV, processor, num_frames=frames), batch_size=1) 
        bleu, latency = evaluate(model, eval_loader, processor)
        
        print(f"      ↳ BLEU: {bleu:.4f} | {latency:.2f}s")
        new_results.append({"Blocks": 12, "Frames": frames, "BLEU4": bleu, "Latency": latency})

    # --- 3. SAFELY CONCATENATE & SAVE ---
    print("\n💾 Safely merging new 12-block data with existing results...")
    new_df = pd.DataFrame(new_results)
    
    if os.path.exists(RESULTS_CSV):
        old_df = pd.read_csv(RESULTS_CSV)
        # Drop any old untrained 12-block rows so we don't have duplicates
        old_df = old_df[old_df['Blocks'] != 12] 
        # Concatenate the preserved 10, 8, 6 block data with our new 12 block data
        final_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        # If the file somehow got deleted, just save the new data
        final_df = new_df
        
    final_df.to_csv(RESULTS_CSV, index=False)
    print(f"✅ All 12-Block tasks finished! {RESULTS_CSV} successfully updated.")

if __name__ == "__main__":
    run_12_block_full()

🚀 Launching 12-Block Train & Evaluate on: mps


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 18038.46it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   ⚙️ Training 12-block model for 2 epochs...


      Epoch 2/2: 100%|██████████| 375/375 [02:43<00:00,  2.29it/s, loss=0.4289]


   ✅ Checkpoint saved to: /Users/ayraj/Desktop/video_captioning/checkpoints_task3/tuned_model_12blocks.pt

   ▶ Evaluating 16 Frames Configuration


      ↳ BLEU: 0.0994 | 0.26s

   ▶ Evaluating 12 Frames Configuration


      ↳ BLEU: 0.1027 | 0.24s

   ▶ Evaluating 8 Frames Configuration


      ↳ BLEU: 0.1143 | 0.22s

💾 Safely merging new 12-block data with existing results...
✅ All 12-Block tasks finished! /Users/ayraj/Desktop/video_captioning/task3_gold_results.csv successfully updated.
